# 03 — Experiments with varying sample sizes

The reference CBN is kept fixed while the number of observations is varied.


In [ ]:
import os
import pandas as pd
import networkx as nx
import pyAgrum as gum
import pyAgrum.lib.notebook as gnb
import matplotlib.pyplot as plt


# ============================================================
# CONFIGURATION
# ============================================================

DATASETS = {
    "Asia": "Asia",
    "Elderly": "Elderly",
    "Suicide": "Suicide"
}

SAMPLE_SIZES = [500, 1000, 5000, 10000, 50000]

OUTPUT_DIR = "learned_structures"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# FUNCTION: SAVE DAG AS IMAGE USING PYAGRUM
# ============================================================

def save_dag_image(edges, nodes, filename, title):
    """
    Convert a list of directed edges into a PyAgrum BN
    and save the resulting structure as an image.
    """

    bn = gum.BayesNet(title)

    # --------------------------------------------------------
    # Add nodes
    # --------------------------------------------------------

    for node in nodes:
        bn.add(
            gum.LabelizedVariable(
                str(node),
                str(node),
                2
            )
        )

    # --------------------------------------------------------
    # Add directed edges
    # --------------------------------------------------------

    for source, target in edges:

        if source in nodes and target in nodes:

            try:
                bn.addArc(
                    str(source),
                    str(target)
                )

            except Exception:
                pass

    # --------------------------------------------------------
    # Save BIF
    # --------------------------------------------------------

    bif_file = filename.replace(".png", ".bif")

    gum.saveBN(bn, bif_file)

    # --------------------------------------------------------
    # Draw network
    # --------------------------------------------------------

    graph = nx.DiGraph()

    graph.add_nodes_from(nodes)
    graph.add_edges_from(edges)

    plt.figure(figsize=(14, 10))

    pos = nx.spring_layout(
        graph,
        seed=42
    )

    nx.draw(
        graph,
        pos,
        with_labels=True,
        node_size=2500,
        arrows=True,
        font_size=10
    )

    plt.title(title)

    plt.tight_layout()

    plt.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved:", filename)

    return bn


# ============================================================
# CAUSALNEX
# ============================================================

def learn_causalnex(data):

    from causalnex.structure.notears import from_pandas

    # NOTEARS
    sm = from_pandas(data)

    edges = [
        (u, v)
        for u, v in sm.edges
    ]

    return edges


# ============================================================
# CAUSAL-LEARN
# ============================================================

def learn_causallearn_pc(data):

    from causallearn.search.ConstraintBased.PC import pc

    # causal-learn expects numerical data
    X = data.astype(float).values

    cg = pc(
        X,
        alpha=0.05
    )

    graph = cg.G

    edges = []

    # causal-learn node indexes start at 0
    columns = list(data.columns)

    for i in range(len(columns)):
        for j in range(len(columns)):

            if i == j:
                continue

            if graph.is_adjacent_to(
                graph.nodes[i],
                graph.nodes[j]
            ):

                # Check orientation
                if graph.get_endpoint(
                    graph.nodes[i],
                    graph.nodes[j]
                ).name == "TAIL":

                    if graph.get_endpoint(
                        graph.nodes[j],
                        graph.nodes[i]
                    ).name == "ARROW":

                        edges.append(
                            (
                                columns[i],
                                columns[j]
                            )
                        )

    return edges


# ============================================================
# CDT
# ============================================================

def learn_cdt_pc(data):

    import cdt

    # CDT expects a pandas DataFrame
    data = data.astype(float)

    pc = cdt.causality.graph.PC()

    graph = pc.create_graph_from_data(
        data
    )

    edges = list(
        graph.edges()
    )

    return edges


# ============================================================
# MAIN EXPERIMENT
# ============================================================

for dataset_name, dataset_prefix in DATASETS.items():

    print("\n")
    print("=" * 70)
    print("DATASET:", dataset_name)
    print("=" * 70)

    for n_samples in SAMPLE_SIZES:

        # ----------------------------------------------------
        # Load generated dataset
        # ----------------------------------------------------

        filename = (
            f"{dataset_prefix}_"
            f"{'SemiSynthetic' if dataset_name != 'Suicide' else 'FullySynthetic'}_"
            f"{n_samples}.csv"
        )

        if not os.path.exists(filename):

            print(
                "Dataset not found:",
                filename
            )

            continue

        data = pd.read_csv(filename)

        print(
            "\nLearning structure:",
            dataset_name,
            "| Samples:",
            n_samples
        )

        # ----------------------------------------------------
        # Make sure data is numeric
        # ----------------------------------------------------

        data = data.apply(
            pd.to_numeric,
            errors="coerce"
        )

        data = data.dropna()

        nodes = list(data.columns)

        # ====================================================
        # 1. CAUSALNEX
        # ====================================================

        try:

            edges = learn_causalnex(data)

            save_dag_image(
                edges,
                nodes,
                (
                    f"{OUTPUT_DIR}/"
                    f"{dataset_name}_"
                    f"{n_samples}_"
                    f"CausalNex.png"
                ),
                (
                    f"{dataset_name} - "
                    f"CausalNex - "
                    f"{n_samples} samples"
                )
            )

        except Exception as e:

            print(
                "CausalNex error:",
                e
            )

        # ====================================================
        # 2. CAUSAL-LEARN
        # ====================================================

        try:

            edges = learn_causallearn_pc(data)

            save_dag_image(
                edges,
                nodes,
                (
                    f"{OUTPUT_DIR}/"
                    f"{dataset_name}_"
                    f"{n_samples}_"
                    f"CausalLearn_PC.png"
                ),
                (
                    f"{dataset_name} - "
                    f"causal-learn PC - "
                    f"{n_samples} samples"
                )
            )

        except Exception as e:

            print(
                "causal-learn error:",
                e
            )

        # ====================================================
        # 3. CDT
        # ====================================================

        try:

            edges = learn_cdt_pc(data)

            save_dag_image(
                edges,
                nodes,
                (
                    f"{OUTPUT_DIR}/"
                    f"{dataset_name}_"
                    f"{n_samples}_"
                    f"CDT_PC.png"
                ),
                (
                    f"{dataset_name} - "
                    f"CDT PC - "
                    f"{n_samples} samples"
                )
            )

        except Exception as e:

            print(
                "CDT error:",
                e
            )


Each learned graph is compared with its expert-defined reference CBN using SHD,
nSHD and edge precision/recall/F1.
